## **EDA Notebook**

**IMPORTANT:** run `setup_data.py`

1. [**EDA Notebook**](#eda-notebook)
   - [Setup](#setup)
     - [Library Imports](#library-imports)
     - [GPU Setup](#gpu-setup)
     - [Project Directory Configuration](#project-directory-configuration)
   - [Data Loading and Exploration](#data-loading-and-exploration)
     - [Dataset Overview](#dataset-overview)
     - [Basic Statistics](#basic-statistics)
     - [Distribution Across Splits](#distribution-across-splits)
   - [Image Visualization](#image-visualization)

### **Setup**

#### **Library Imports**

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
sys.path.append('../')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from src.utils.visualizations import *
from src.utils.preproc import *
from setup_data import update_image_paths

from numba import cuda
import tensorflow as tf

### **GPU Setup**

In [ ]:
def clear_gpu_memory():
    device = cuda.get_current_device()
    device.reset() # dump the memory contents to free up the memory (it accumulates over the session)

# CUDA (Nvidia GPU Computing)
if len(tf.config.list_physical_devices('GPU')) > 0:
    gpus = tf.config.list_physical_devices('GPU')
    print("Num GPUs Available: ", len(gpus))
    
    clear_gpu_memory()
    
    tf.config.experimental.set_memory_growth(gpus[0], True)

    tf.config.set_logical_device_configuration(
    gpus[0], 
    [tf.config.LogicalDeviceConfiguration(memory_limit=8192)])  # limit to 4GB

    tf.compat.v1.disable_eager_execution()

### **Project Directory Configuration**

In [ ]:
# because the utils in the src are designed to be run from the root of the project,
# and by default jupyter runs from the notebook directory we need to change the working directory to the root

def find_project_root(filename=".git"): # .git is located in the root of the project
    current_dir = os.getcwd()
    while current_dir != os.path.dirname(current_dir): # stops only when at the root (moves up 1 level each iteration)
        if filename in os.listdir(current_dir):
            return current_dir
        current_dir = os.path.dirname(current_dir)

project_root = find_project_root()
os.chdir(project_root)  # change the working directory to the project root

print("Project root:", project_root, "CWD:", os.getcwd())

### **Data Loading and Exploration**

#### **Dataset Overview**

In [ ]:
img_metadata_df = pd.read_csv('image_metadata/image_data.csv')
img_metadata_df.isnull().sum()

In [ ]:
img_metadata_df.dropna(inplace=True)
img_metadata_df.head(3)

In [ ]:
img_metadata_df.isnull().sum()

In [ ]:
def get_image_location(row):
    # because the path contains the split, we can use it to create a new column for further analysis
    possible_locations = ['train', 'test', 'val']
    for location in possible_locations:
        if location in row['path_to_image']:
            return location
    return "NOT FOUND"

img_metadata_df = update_image_paths(img_metadata_df)

img_metadata_df['image_location'] = img_metadata_df.apply(get_image_location, axis=1)
img_metadata_df.head(3)

<i> The updated version of the image metadata csv files contains the actual image paths, and the missing images have been removed. </i>

In [ ]:
resolutions = check_image_resolutions(img_metadata_df)
resolutions_df = pd.DataFrame(resolutions, columns=['width', 'height'])
print("\nImage Resolutions:\n", resolutions_df.describe())

In [ ]:
resolutions_df['height'].value_counts().sort_index()

#### **Basic Statistics**

In [ ]:
# Overall Image Distribution

grouped = img_metadata_df.groupby(
    ['Benign or Malignant', 'Cancer Type', 'Magnification']
).size().reset_index(name='Count')

pivot_table = grouped.pivot_table(
    index=['Benign or Malignant', 'Cancer Type'],
    columns='Magnification',
    values='Count',
    aggfunc='sum',
    fill_value=0
)

magnification_order = ['40X', '100X', '200X', '400X']
pivot_table = pivot_table.reindex(columns=magnification_order, fill_value=0)

pivot_table['Total'] = pivot_table.sum(axis=1)

pivot_table = pivot_table.astype(int) # remove decimal points
pivot_table

<i>Notes :<br> 
- The binary classification <b>(Benign or Malignant) is unbalanced.</b><br>
- The <b>cancer types</b> within each binary class are <b>also unbalanced</b> (especially D.U (Malignant)).<br>
- Their <b>distributions</b> are <b>similar</b> in <b>all magnifications</b> though.</i>

In [ ]:
# pivot table to check the distribution of images by magnification and Benign or Malignant

distribution_table = img_metadata_df.pivot_table(
    index='Magnification',
    columns='Benign or Malignant',
    values='path_to_image',
    aggfunc='count',
    fill_value=0
)

distribution_table['Total'] = distribution_table.sum(axis=1)
distribution_table.loc['Total'] = distribution_table.sum(axis=0)

distribution_table

In [ ]:
sns.countplot(data=img_metadata_df, x='Magnification', order=['40X', '100X', '200X', '400X'])
plt.title("Image Distribution by Magnification")
plt.show()

<i> Notes:<br> 
- Like the previous case, the <b>distributions</b> are <b>similar in all magnifications.</b></i>

#### **Distribution Across Splits**

In [ ]:
plot_distribution_pie(img_metadata_df, 'Benign or Malignant')

In [ ]:
plot_distribution_pie(img_metadata_df, 'Cancer Type')

**Confirmation of the same distribution across splits (stratification was used)**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 6))

splits = ['train', 'val', 'test']
for ax, split in zip(axes, splits):
    plot_distribution_pie(img_metadata_df[img_metadata_df['image_location'] == split], 'Benign or Malignant', ax=ax)
    ax.set_title(f'Distribution in {split} split')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

splits = ['train', 'val', 'test']
for ax, split in zip(axes, splits):
    plot_distribution_pie(img_metadata_df[img_metadata_df['image_location'] == split], 'Cancer Type', ax=ax)
    ax.set_title(f'Distribution in {split} split')

plt.tight_layout()
plt.show()

#### **Image Visualization**

In [ ]:
def display_example_images(paths, title="Example Images"):
    plt.figure(figsize=(15, 10))
    for i, path in enumerate(paths):
        plt.subplot(1, len(paths), i + 1)
        with Image.open(path) as img:
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"{title} {i+1}")
    plt.tight_layout()
    plt.show()

cancer_types = img_metadata_df['Cancer Type'].unique()
for cancer_type in cancer_types[:3]:
    subset = img_metadata_df[img_metadata_df['Cancer Type'] == cancer_type]
    example_paths = subset['path_to_image'].sample(3).tolist()
    display_example_images(example_paths, title=f"{cancer_type}")

In [ ]:
magnifications = ['40X', '100X', '200X', '400X']
cancer_types = list(img_metadata_df['Cancer Type'].unique())
            
plot_images_compare_magnification(img_metadata_df, cancer_types, magnifications)

In [ ]:
benign_types = ['Adenosis', 'Fibroadenoma', 'Phyllodes Tumor', 'Tubular Adenoma']
malignant_types = ['Ductal Carcinoma', 'Lobular Carcinoma', 'Mucinous Carcinoma', 'Papillary Carcinoma']

plot_images_cancer_types(img_metadata_df, benign_types=benign_types, malignant_types=malignant_types, magnifications=['100X'])

In [ ]:
datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest')

X_train, y_train, X_test, y_test, X_val, y_val = load_and_preprocess_data('image_metadata/updated_image_data.csv', (224, 224), 'Cancer Type')
visualize_augmented_images(X_train, y_train, datagen, 5)